### Construct a model (Example: nonlinear)
https://brian2.readthedocs.io/en/latest/examples/synapses.nonlinear.html

In [ ]:
import brian2 as b2
import matplotlib.pyplot as plt

from neuroseq.b2_model import B2Model

In [ ]:
def create_model() -> B2Model:
    a = 1 / (10 * b2.ms)
    b = 1 / (10 * b2.ms)
    c = 1 / (10 * b2.ms)

    model = B2Model(seed=4321)

    model.define_parameter("a", a)
    model.define_parameter("b", b)
    model.define_parameter("c", c)

    model.define_neuronnumber("neuron_input", 2)
    model.define_solver_method("neuron_input", "euler")
    model.define_equation("neuron_input", "dv/dt = 1/(10*ms) : 1")
    model.define_threshold("neuron_input", "v>1")
    model.define_reset("neuron_input", "v=0")

    model.define_neuronnumber("neurons", 1)
    model.define_equation(
        "neurons",
        """dv/dt = (g-v)/(10*ms) : 1
                            g : 1""",
    )
    model.define_solver_method("neurons", "exact")
    model._assemble_neurongroup("neurons")
    model._assemble_neurongroup("neuron_input")

    synapse_eq = """
        dg_syn/dt = -a*g_syn+b*x*(1-g_syn) : 1 (clock-driven)
        g_post = g_syn : 1 (summed)
        dx/dt=-c*x : 1 (clock-driven)
        w : 1 # synaptic weight
    """

    model.define_synapse_equation("S", synapse_eq)
    model.define_synapse_origin("S", "neuron_input")
    model.define_synapse_target("S", "neurons")
    model.define_synapse_pre("S", "x += w")
    model._assemble_synapse("S")

    model.define_neuronsubgroup_target("input_subgroup", "neuron_input")
    model.define_neuronsubgroup_idxs("input_subgroup", 1)
    model._assemble_neuronsubgroup("input_subgroup")

    model.define_monitor_source("M", "S")
    model.define_monitor_type("M", "StateMonitor")
    model.define_monitor_variable("M", "g")
    model._assemble_monitor("M")

    model.define_monitor_source("Mn", "neurons")
    model.define_monitor_type("Mn", "StateMonitor")
    model.define_monitor_variable("Mn", "g")
    model._assemble_monitor("Mn")

    model.define_init("S", "w", [1.0, 10.0])
    model.define_init("neuron_input", "v", [0.0, 0.5])

    for n in model.initials:
        model._assemble_initial_conditions(n)

    model.must_be_rebuilt = False
    return model

In [ ]:
model = create_model()
model.assemble_network()

In [ ]:
test = model.run_simulation(2 * b2.second)

In [ ]:
M = model.get_monitor("M")["object"]
Mn = model.get_monitor("Mn")["object"]

In [ ]:
plt.subplot(221)

plt.subplot(2, 1, 1)
plt.plot(M.t / b2.ms, M.g.T)
plt.xlabel("Time (ms)")
plt.ylabel("g_syn")
plt.subplot(2, 1, 2)
plt.plot(Mn.t / b2.ms, Mn[0].g)
plt.ylabel("Time (ms)")
plt.ylabel("g")
plt.tight_layout()
plt.show()

In [ ]:
model.save_model(
    r"C:\Users\af24\Documents\GitHub\neuroseq\src\neuroseq\example_models\example_non-linear.nseq"
)